# HairNet Compliance Detection

### Test Set Evaluation

For each of the 8 trained models:
1. Looks up the MLflow run by name
2. Downloads `best.pt` from MLflow artifacts
3. Evaluates on the **test split** (47 images — never seen during training)
4. Logs `test/*` metrics back into the same MLflow run
5. Prints a final comparison table: val mAP50 vs test mAP50

## Install Dependencies

In [1]:
!pip install ultralytics mlflow roboflow python-dotenv -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports & Setup

In [5]:
import os
import mlflow
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

# Disable Ultralytics built-in MLflow so it doesn't interfere
from ultralytics import settings
settings.update({'mlflow': False})

MLFLOW_TRACKING_URI = 'mlruns'
EXPERIMENT_NAME = 'hairnet-compliance-detection'
DATA_YAML = Path('HairNet-Compliance-Detection-1/data.yaml')

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

RUN_NAMES = [
    'yolov26n-baseline',
    'yolov26m-baseline',
    'yolo12n-baseline',
    'yolo12m-baseline',
    'yolov26n-augmented',
    'yolov26m-augmented',
    'yolo12n-augmented',
    'yolo12m-augmented',
]

print(f'MLflow URI  : {mlflow.get_tracking_uri()}')
print(f'Experiment  : {EXPERIMENT_NAME}')
print(f'Data YAML   : {DATA_YAML}')
print(f'Runs to eval: {len(RUN_NAMES)}')

MLflow URI  : mlruns
Experiment  : hairnet-compliance-detection
Data YAML   : HairNet-Compliance-Detection-1\data.yaml
Runs to eval: 8


## 2. Test Evaluation Loop

For each run:
- Looks up run ID by name from MLflow
- Downloads `best.pt` from MLflow artifact store
- Runs `model.val(split='test')` on the 47 test images
- Appends `test/*` metrics to the existing MLflow run

In [7]:
from urllib.parse import urlparse
from urllib.request import url2pathname

results_summary = []

for run_name in RUN_NAMES:
    print(f'\n{"-"*55}')
    print(f'Evaluating: {run_name}')

    # ── 1. Look up run by name
    runs_df = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string=f"tags.`mlflow.runName` = '{run_name}'",
    )
    if runs_df.empty:
        print(f'  [SKIP] No run found with name "{run_name}"')
        continue

    run_id = runs_df.iloc[0]['run_id']
    print(f'  Run ID : {run_id}')

    # ── 2. Resolve best.pt 
    try:
        run_info   = mlflow.get_run(run_id)
        exp_id     = run_info.info.experiment_id          
        best_pt_path = Path('mlruns') / exp_id / run_id / 'artifacts' / 'weights' / 'best.pt'
        if not best_pt_path.exists():
            print(f'  [SKIP] best.pt not found at {best_pt_path}')
            continue
        print(f'  Weights: {best_pt_path}')
    except Exception as e:
        print(f'  [SKIP] Could not resolve best.pt: {e}')
        continue

    # ── 3. Evaluate on test split
    model = YOLO(best_pt_path)
    test_results = model.val(
        data=str(DATA_YAML),
        split='test',
        verbose=False,
    )
    m = test_results.results_dict

    test_metrics = {
        'test/mAP50'    : m.get('metrics/mAP50(B)',     0),
        'test/mAP50-95' : m.get('metrics/mAP50-95(B)', 0),
        'test/precision': m.get('metrics/precision(B)', 0),
        'test/recall'   : m.get('metrics/recall(B)',    0),
        'test/box_loss' : m.get('val/box_loss',         0),
        'test/cls_loss' : m.get('val/cls_loss',         0),
    }

    # ── 4. Log back into the existing run
    mlflow.end_run()
    with mlflow.start_run(run_id=run_id):
        mlflow.log_metrics(test_metrics)

    print(f'  test/mAP50     : {test_metrics["test/mAP50"]:.4f}')
    print(f'  test/recall    : {test_metrics["test/recall"]:.4f}')
    print(f'  test/precision : {test_metrics["test/precision"]:.4f}')
    print(f'  test/mAP50-95  : {test_metrics["test/mAP50-95"]:.4f}')

    results_summary.append({'run_name': run_name, 'run_id': run_id, **test_metrics})

print(f'\n{"="*55}')
print(f'Done — evaluated {len(results_summary)} / {len(RUN_NAMES)} runs')


-------------------------------------------------------
Evaluating: yolov26n-baseline
  Run ID : 0a2abe883e56454e8a687b66e8e35687
  Weights: mlruns\242006421917660016\0a2abe883e56454e8a687b66e8e35687\artifacts\weights\best.pt
Ultralytics 8.4.50  Python-3.12.6 torch-2.11.0+cpu CPU (11th Gen Intel Core(TM) i7-11390H 3.40GHz)
YOLO26n summary (fused): 122 layers, 2,375,421 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2209.3548.7 MB/s, size: 705.4 KB)
val: Scanning C:\Users\wajee\PycharmProjects\HairNet Compliance Detection\HairNet-Compliance-Detection-1\test\labels... 47 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 47/47 2.1Kit/s 0.0s
val: New cache created: C:\Users\wajee\PycharmProjects\HairNet Compliance Detection\HairNet-Compliance-Detection-1\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.5s/it 4.6s3.1ss
                   all         47        704      0

## 3. Summary Table

All 8 models ranked by `test/mAP50` (highest = best on unseen data).

In [8]:
if results_summary:
    df = pd.DataFrame(results_summary)
    df = df.sort_values('test/mAP50', ascending=False).reset_index(drop=True)
    df.index += 1  # rank starts at 1

    display_cols = ['run_name', 'test/mAP50', 'test/mAP50-95', 'test/precision', 'test/recall', 'test/box_loss', 'test/cls_loss']
    print(df[display_cols].to_string(float_format='{:.4f}'.format))

    best = df.iloc[0]
    print(f'\nBest model : {best["run_name"]}')
    print(f'  test/mAP50  : {best["test/mAP50"]:.4f}')
    print(f'  test/recall : {best["test/recall"]:.4f}')
else:
    print('No results to display — check that run names match and best.pt artifacts exist.')

             run_name  test/mAP50  test/mAP50-95  test/precision  test/recall  test/box_loss  test/cls_loss
1   yolo12m-augmented      0.9943         0.8306          0.9901       0.9834              0              0
2  yolov26m-augmented      0.9927         0.8271          0.9794       0.9833              0              0
3   yolo12n-augmented      0.9926         0.7980          0.9876       0.9817              0              0
4    yolo12n-baseline      0.9915         0.8289          0.9906       0.9731              0              0
5    yolo12m-baseline      0.9908         0.8656          0.9920       0.9754              0              0
6   yolov26m-baseline      0.9876         0.8586          0.9879       0.9567              0              0
7  yolov26n-augmented      0.9781         0.7566          0.9424       0.9202              0              0
8   yolov26n-baseline      0.9607         0.7826          0.9443       0.8605              0              0

Best model : yolo12m-augmen